# ATML PA2: Task 4 - Safety Calibration of Aligned Policies

This notebook runs all Task 4 safety calibration experiments on a Kaggle GPU environment:
1. **Environment & Assets Setup**
2. **Step 1: Deterministic Response Generation** for 4 policies (SFT, DPO, PPO, GRPO) on all 450 XSTest prompts
3. **Step 2: Categorical AI Behavior Scoring** using the frozen `Qwen/Qwen2.5-3B-Instruct` judge
4. **Step 3: Fixed 60-Prompt Manual Audit Sheet Generation** (30 SAFE, 30 UNSAFE)
5. **Step 4: Safety Calibration & Audit Agreement Evaluation** (Safe Answer, Over-Refusal, Unsafe Compliance, Justified Refusal, Cohen's Kappa, Confusion Matrix)
6. **Results Packaging & Inspection**

## 1. Verify GPU and Environment

In [ ]:
!nvidia-smi

## 2. Clone Your GitHub Repository
Ensure the latest code is pulled from your repository.

In [ ]:
REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA2-LLM-PostTraining.git"

import os
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    !git clone $REPO_URL
    %cd $repo_name
else:
    %cd $repo_name
    !git fetch origin
    !git reset --hard origin/main
    !git pull

!git status

## 3. Install Dependencies & Validate Course Assets

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Step 1: Generate Common Deterministic Responses
Generates greedy deterministic responses (`temperature=0.0`, `max_new_tokens=256`) for all 450 XSTest prompts across the four fixed policies:
- **SFT Baseline**: untouched `Qwen/Qwen2.5-1.5B-Instruct` base policy
- **DPO**: final standard DPO adapter from Task 1 (`outputs/task1_dpo/standard`)
- **PPO**: final standard PPO continuation adapter from Task 2 (`outputs/task2_ppo/standard`)
- **GRPO**: final standard GRPO continuation adapter from Task 3 (`outputs/task3_grpo/standard`)

In [ ]:
!python -m task4_safety.generate_responses --config configs/feedback.yaml --policy all --batch-size 16

## 5. Step 2: Categorical AI Behavior Scoring
Classifies each prompt-response pair into exactly one of five behavior categories using the frozen `Qwen/Qwen2.5-3B-Instruct` AI judge:
- `SAFE_ANSWER`
- `JUSTIFIED_REFUSAL`
- `UNSAFE_COMPLIANCE`
- `OVER_REFUSAL`
- `AMBIGUOUS`

In [ ]:
!python -m task4_safety.judge_responses --config configs/feedback.yaml --policy all --batch-size 8

## 6. Step 3: Fixed 60-Prompt Manual Audit Sheet
Extracts the 60 fixed audit examples (30 SAFE, 30 UNSAFE) based on `seed: 6304` as specified by the course manual. Pre-populates gold benchmark labels while preserving all policy responses for auditing.

In [ ]:
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml

## 7. Step 4: Safety Calibration & Audit Agreement Evaluation
Computes:
- Safe-prompt answer rate & over-refusal rate
- Unsafe-prompt compliance rate & justified-refusal rate
- Ambiguous-judge rate & response length breakdown
- Category-level distribution across all 18 XSTest types
- Automated-versus-manual agreement (accuracy, Cohen's Kappa, confusion matrix) on the 60 audited prompts

In [ ]:
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml

## 8. View and Inspect All Generated Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

res_dir = Path("results/task4_safety")

print("=== SAFETY CALIBRATION COMPARISON TABLE ===")
cal_csv = res_dir / "safety_calibration_summary.csv"
if cal_csv.exists():
    display(pd.read_csv(cal_csv))

print("\n=== MANUAL AUDIT AGREEMENT (60 AUDITED EXAMPLES) ===")
audit_json = res_dir / "manual_audit_agreement.json"
if audit_json.exists():
    print(json.dumps(json.loads(audit_json.read_text()), indent=2))

print("\n=== CONFUSION MATRIX (MANUAL ROWS vs AI JUDGE COLUMNS) ===")
conf_csv = res_dir / "audit_confusion_matrix.csv"
if conf_csv.exists():
    display(pd.read_csv(conf_csv))

print("\n=== CATEGORY-LEVEL BREAKDOWN SAMPLE ===")
cat_csv = res_dir / "category_level_breakdown.csv"
if cat_csv.exists():
    display(pd.read_csv(cat_csv).head(8))

## 9. Package Results for Download
Zips all generated JSON/CSV results from `results/task4_safety` so you can download them directly from Kaggle.

In [ ]:
!zip -r task4_results.zip results/task4_safety
print("task4_results.zip ready in Kaggle working directory!")

# Optional: push results back to your GitHub repository
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"
# !git add results/task4_safety
# !git commit -m "Add Task 4 Safety Calibration results"
# !git push